# Azure ML Model Deployment — SDK v2

This notebook deploys a diabetes prediction model to an **Azure Machine Learning Managed Online Endpoint** using the **Azure AI ML Python SDK v2** (`azure-ai-ml`).

In [ ]:
from dotenv import load_dotenv
import os

from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    Environment,
    CodeConfiguration,
)
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

In [ ]:
load_dotenv()

subscription_id = os.getenv("SUBSCRIPTION_ID")
resource_group = os.getenv("RESOURCE_GROUP")
workspace_name = os.getenv("WORKSPACE_NAME")
tenant_id= os.getenv("TENANT_ID")
region_name = os.getenv("REGION")

In [ ]:
from azure.ai.ml.entities import Workspace

try:
    credential = DefaultAzureCredential(tenant_id=tenant_id)
    credential.get_token("https://management.azure.com/.default")
except Exception:
    # Fall back to interactive browser login for local development
    credential = InteractiveBrowserCredential(tenant_id=tenant_id)

ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)

In [ ]:
print("Creating workspace...")
workspace = ml_client.workspaces.begin_create(
    Workspace(
        name=workspace_name,
        location=region_name,
        display_name="Diabetes ML Model Workspace",
        description="Workspace for diabetes prediction model",
    )
).result()

In [ ]:
model_path = "./model/model.pkl"
model_name = "diabetes_ml_model"

model = Model(
    path=model_path,
    name=model_name,
    description="Diabetes prediction model",
    type="custom_model",
)

registered_model = ml_client.models.create_or_update(model)

In [ ]:
ENVIRONMENT_FILE = os.path.join(".", "environment.yml")

env = Environment(
    name="diabetes-prediction-env",
    description="Conda environment for the diabetes scoring service",
    conda_file=ENVIRONMENT_FILE,
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
)

In [ ]:
from azure.mgmt.resource import ResourceManagementClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
import time

try:
    credential = DefaultAzureCredential(tenant_id=tenant_id)
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential(tenant_id=tenant_id)

resource_client = ResourceManagementClient(credential, subscription_id)

required_providers = [
    "Microsoft.MachineLearningServices",
    "Microsoft.ContainerRegistry",
    "Microsoft.ContainerInstance",
    "Microsoft.Storage",
    "Microsoft.KeyVault",
    "Microsoft.Network",
    "Microsoft.Compute",
    "Microsoft.OperationalInsights",
    "Microsoft.Insights",
]

# Register any unregistered providers
print("Checking provider registration states...")
for provider in required_providers:
    state = resource_client.providers.get(provider).registration_state
    if state != "Registered":
        print(f"  ⏳ Registering {provider} (was: {state})...")
        resource_client.providers.register(provider)
    else:
        print(f"  ✅ {provider} already registered")

# Wait until all confirmed
print("\nWaiting for all providers to confirm...")
while True:
    states = {p: resource_client.providers.get(p).registration_state for p in required_providers}
    unregistered = {p: s for p, s in states.items() if s != "Registered"}
    
    if not unregistered:
        print("✅ All providers registered — safe to proceed")
        break
    
    for p, s in unregistered.items():
        print(f"  ⏳ {p}: {s}")
    print("Waiting 20s...\n")
    time.sleep(20)

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%m%d%H%M")
endpoint_name = os.getenv("ENDPOINT_NAME")     

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Online endpoint for diabetes prediction service",
    auth_mode="key",
)

In [ ]:
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

In [ ]:
deployment_name = "blue"

deployment = ManagedOnlineDeployment(
    name=deployment_name,
    endpoint_name=endpoint_name,
    model=registered_model,
    environment=env,
    code_configuration=CodeConfiguration(code=".", scoring_script="score.py"),
    instance_type="Standard_F2s_v2",
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(deployment).result()

In [ ]:
# Send 100% of traffic to the blue deployment
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint=endpoint).result()
print("Traffic routing updated: 100% -> blue")

In [ ]:
ml_client.online_endpoints.get(name=endpoint_name)

In [ ]:
import json
import requests

load_dotenv()

input_data = {"data": [[1.0,1.0,1.0,29.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,1.0,11.0,6.0,7.0]]}

scoring_uri = os.getenv("SCORING_URI")
api_key = os.getenv("API_KEY")

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}"
}

response = requests.post(
    url=scoring_uri,
    data=json.dumps(input_data),
    headers=headers
)

if response.status_code == 200:
    result = response.json()  
    print(result)
else:
    print(f"Error {response.status_code}: {response.text}")